# Beehive Audio Preprocessing Pipeline

This notebook refines the preprocessing stage of the beehive acoustic monitoring project into a cleaner, GitHub-ready workflow.

## Goals
- load raw `.wav` recordings safely
- apply **peak normalization**
- apply a **band-pass filter** focused on bee-relevant frequencies
- **resample** recordings to a lower sample rate for faster downstream analysis
- save processed files in a reproducible folder structure
- generate a simple processing summary for quality control

## Why this version is stronger for a portfolio
- clearer function structure
- parameterized configuration
- basic validation and error handling
- reproducible batch processing
- cleaner markdown explanations for future readers


## 1. Imports

The pipeline uses:
- `pathlib` for safer path handling
- `librosa` for audio loading and resampling
- `soundfile` for writing `.wav` files
- `scipy.signal` for digital filtering
- `pandas` for a processing log
- `matplotlib` for quick visual inspection


In [ ]:
from pathlib import Path
import warnings

import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, sosfiltfilt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 2. Configuration

Edit these paths and parameters before running the notebook on a new dataset.


In [ ]:
# -------- USER CONFIG --------
INPUT_FOLDER = Path(r"X:/Raw Sets of Bee Recordins/SET III Bartek/30.06.2025 - 01.07.2025")
OUTPUT_FOLDER = Path(r"X:/Dissertation Files/Preprocessed files Set III 30.06.2025 - 01.07.2025")

NORMALIZE = True
APPLY_FILTER = True
APPLY_RESAMPLE = True

TARGET_SR = 16000
LOWCUT = 100
HIGHCUT = 3000
FILTER_ORDER = 5

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

## 3. Preprocessing design

### 3.1 Loading strategy
Each file is processed independently instead of concatenating all recordings into one long signal.  
This makes the workflow:
- easier to debug
- less memory intensive
- easier to rerun if one file fails

### 3.2 Peak normalization
Peak normalization scales the waveform relative to its largest absolute amplitude.  
This improves consistency across recordings, although it should be noted in the dissertation that normalization may reduce information about absolute recording level.

### 3.3 Band-pass filtering
A band-pass filter is used to reduce irrelevant low-frequency rumble and high-frequency environmental noise.  
For this project the working range is **100 Hz to 3000 Hz**, which is a practical range for retaining much of the bee-related acoustic content while removing some background interference.

### 3.4 Resampling
The original recordings are large. Resampling to **16 kHz** reduces storage and speeds up later feature extraction while still preserving the frequency range of interest for this analysis.


## 4. Core functions

In [ ]:
def bandpass_filter(signal: np.ndarray,
                    sr: int,
                    lowcut: float = 100,
                    highcut: float = 3000,
                    order: int = 5) -> np.ndarray:
    """Apply a Butterworth band-pass filter using second-order sections.

    Parameters
    ----------
    signal : np.ndarray
        Input mono waveform.
    sr : int
        Sampling rate of the input signal.
    lowcut : float
        Lower cutoff frequency in Hz.
    highcut : float
        Upper cutoff frequency in Hz.
    order : int
        Filter order.

    Returns
    -------
    np.ndarray
        Filtered waveform.
    """
    if lowcut <= 0 or highcut >= sr / 2:
        raise ValueError(f"Invalid cutoff frequencies for sr={sr}.")
    if lowcut >= highcut:
        raise ValueError("lowcut must be smaller than highcut.")

    sos = butter(order, [lowcut, highcut], btype="bandpass", fs=sr, output="sos")
    return sosfiltfilt(sos, signal)


def peak_normalize(signal: np.ndarray) -> np.ndarray:
    """Peak-normalize an audio signal safely."""
    peak = np.max(np.abs(signal))
    if peak == 0:
        return signal
    return signal / peak


def preprocess_audio_file(input_path: Path,
                          output_path: Path | None = None,
                          normalize: bool = True,
                          apply_filter: bool = True,
                          apply_resample: bool = True,
                          target_sr: int = 16000,
                          lowcut: float = 100,
                          highcut: float = 3000,
                          filter_order: int = 5) -> tuple[np.ndarray, int, dict]:
    """Load, preprocess, and optionally save one audio file.

    Returns
    -------
    tuple
        (processed_signal, processed_sample_rate, metadata_dict)
    """
    signal, sr = librosa.load(input_path, sr=None, mono=True)

    original_duration = len(signal) / sr
    original_peak = float(np.max(np.abs(signal))) if len(signal) else np.nan

    if normalize:
        signal = peak_normalize(signal)

    if apply_filter:
        signal = bandpass_filter(signal, sr, lowcut=lowcut, highcut=highcut, order=filter_order)

    if apply_resample and sr != target_sr:
        signal = librosa.resample(signal, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    if output_path is not None:
        sf.write(output_path, signal, sr)

    metadata = {
        "file_name": input_path.name,
        "original_sr": None,  # filled below
        "processed_sr": sr,
        "duration_seconds": original_duration,
        "original_peak": original_peak,
        "processed_peak": float(np.max(np.abs(signal))) if len(signal) else np.nan,
        "num_samples_after": len(signal),
        "status": "success"
    }
    return signal, sr, metadata

In [ ]:
def process_folder(input_folder: Path,
                   output_folder: Path,
                   normalize: bool = True,
                   apply_filter: bool = True,
                   apply_resample: bool = True,
                   target_sr: int = 16000,
                   lowcut: float = 100,
                   highcut: float = 3000,
                   filter_order: int = 5) -> pd.DataFrame:
    """Batch-process all WAV files in a folder and return a log DataFrame."""
    rows = []

    wav_files = sorted(input_folder.glob("*.wav"))
    if not wav_files:
        raise FileNotFoundError(f"No WAV files found in: {input_folder}")

    for input_path in wav_files:
        output_path = output_folder / input_path.name

        try:
            original_info = sf.info(str(input_path))

            signal, sr, metadata = preprocess_audio_file(
                input_path=input_path,
                output_path=output_path,
                normalize=normalize,
                apply_filter=apply_filter,
                apply_resample=apply_resample,
                target_sr=target_sr,
                lowcut=lowcut,
                highcut=highcut,
                filter_order=filter_order
            )

            metadata["original_sr"] = original_info.samplerate
            metadata["original_frames"] = original_info.frames
            metadata["original_channels"] = original_info.channels
            metadata["original_subtype"] = original_info.subtype
            metadata["output_path"] = str(output_path)
            rows.append(metadata)

        except Exception as exc:
            rows.append({
                "file_name": input_path.name,
                "status": "failed",
                "error": str(exc)
            })

    return pd.DataFrame(rows)

## 5. Run batch preprocessing

In [ ]:
processing_log = process_folder(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    normalize=NORMALIZE,
    apply_filter=APPLY_FILTER,
    apply_resample=APPLY_RESAMPLE,
    target_sr=TARGET_SR,
    lowcut=LOWCUT,
    highcut=HIGHCUT,
    filter_order=FILTER_ORDER
)

processing_log.head()

## 6. Quality-control summary

This table helps verify that files were processed successfully and that the sample rate conversion worked as expected.


In [ ]:
processing_log["status"].value_counts(dropna=False)

In [ ]:
processing_log.describe(include="all")

## 7. Save the processing log

Saving a CSV log is useful for reproducibility and for documenting the preprocessing stage in the dissertation and GitHub repository.


In [ ]:
log_path = OUTPUT_FOLDER / "preprocessing_log.csv"
processing_log.to_csv(log_path, index=False)
log_path

## 8. Visual comparison: raw vs preprocessed signal

This quick plot is only for visual inspection. It helps confirm that the processed waveform differs from the raw version in a sensible way.


In [ ]:
def plot_waveform_comparison(raw_path: Path,
                             processed_path: Path,
                             max_seconds: float = 10.0) -> None:
    raw_signal, raw_sr = librosa.load(raw_path, sr=None, mono=True)
    processed_signal, processed_sr = librosa.load(processed_path, sr=None, mono=True)

    raw_n = int(min(len(raw_signal), max_seconds * raw_sr))
    processed_n = int(min(len(processed_signal), max_seconds * processed_sr))

    raw_time = np.arange(raw_n) / raw_sr
    processed_time = np.arange(processed_n) / processed_sr

    plt.figure(figsize=(14, 4))
    plt.plot(raw_time, raw_signal[:raw_n], label="Raw")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title("Raw waveform (first segment)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 4))
    plt.plot(processed_time, processed_signal[:processed_n], label="Preprocessed")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title("Preprocessed waveform (first segment)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
first_success = processing_log.loc[processing_log["status"] == "success", "file_name"].iloc[0]
raw_example = INPUT_FOLDER / first_success
processed_example = OUTPUT_FOLDER / first_success

plot_waveform_comparison(raw_example, processed_example, max_seconds=10)

## 9. Suggested GitHub notes

When you publish this notebook, mention these points in the repository README:

1. **Project purpose**  
   This preprocessing stage prepares raw hive recordings for downstream feature extraction and exploratory analysis.

2. **Processing choices**  
   - mono loading  
   - peak normalization  
   - 100–3000 Hz band-pass filtering  
   - resampling to 16 kHz

3. **Limitations**  
   - peak normalization may remove absolute loudness differences between recordings  
   - the chosen frequency band is a practical engineering choice and should be justified with literature where possible  
   - further denoising was not applied here to avoid overprocessing biologically relevant content

4. **Reproducibility**  
   - keep raw data unchanged  
   - save processed outputs separately  
   - export a processing log CSV


## 10. Portfolio improvement ideas

For the next refinement pass, you could extend this notebook with:
- spectrogram comparison before vs after preprocessing
- automatic clipping / silence diagnostics
- file-size comparison
- unit-style validation checks for sample rate and output count
- a reusable `.py` module version of the same pipeline for cleaner repository structure
